In [ ]:
import os
os.environ['HF_HOME'] = '/workspace/persistent'

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch
from huggingface_hub import login

# push to hf



# Log in to Hugging Face
login(token="HF_TOKEN_PLACEHOLDER")

model_name = "meta-llama/Meta-Llama-3.1-70B-Instruct"

# Configure 4-bit quantization with the storage type included
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Load in 4-bit precision
    bnb_4bit_use_double_quant=True,  # Optional: Use double quantization
    bnb_4bit_quant_type="nf4",  # Use NormalFloat4 (NF4) quantization type
    bnb_4bit_compute_dtype=torch.float16,  # Use float16 for computation
    bnb_4bit_quant_storage=torch.float16  # Storage dtype (this must match your training setting)
)

# Load the base model with 4-bit quantization
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quantization_config
)

# Load the LoRA weights
peft_model = PeftModel.from_pretrained(
    base_model,  # Pass the base model object here
    "./llama-sft-lora-fsdp"  # Directory where LoRA weights are saved
)

# Push the LoRA weights to Hugging Face
peft_model.push_to_hub("sep_22_v2Storage_70B_Q4_512seqLen_16bch_20steps_16r32a_loss0-06_layersAll-Linear_lr1e-3")
